![Two data scientists working on a dashboard.](hr-image-small.png)

Un problème courant lorsqu’on crée des modèles pour générer de la valeur métier à partir des données est que les jeux de données peuvent être si volumineux que le modèle met des jours à produire des prédictions. S’assurer que votre jeu de données est stocké de la manière la plus efficace possible est essentiel pour permettre à ces modèles de s’exécuter dans des délais raisonnables, sans devoir en réduire la taille.

Vous avez été recruté par un grand organisme de formation en data science en ligne, *Training Data Ltd.*, pour optimiser l’un de ses plus gros jeux de données clients. Ce jeu de données sera ensuite utilisé pour prédire si leurs apprenants cherchent un nouvel emploi ou non, une information qui leur permettra de les mettre en relation avec des recruteurs potentiels.

Vous avez accès à `customer_train.csv`, un sous-ensemble de l’ensemble complet de leurs clients, afin de réaliser une preuve de concept d’une solution de stockage bien plus efficace. Le jeu de données contient des informations anonymisées sur les apprenants, ainsi qu’un indicateur précisant s’ils recherchaient un nouvel emploi pendant la formation :

| Column                   | Description                                                                      |
|------------------------- |--------------------------------------------------------------------------------- |
| `student_id`             | Identifiant unique de chaque apprenant.                                          |
| `city`                   | Code de la ville où vit l’apprenant.                                             |
| `city_development_index` | Indice de développement de la ville (échelle normalisée).                         |
| `gender`                 | Genre de l’apprenant.                                                            |
| `relevant_experience`    | Indicateur de l’expérience professionnelle pertinente de l’apprenant.            |
| `enrolled_university`    | Type de cursus universitaire suivi (le cas échéant).                             |
| `education_level`        | Niveau d’études de l’apprenant.                                                  |
| `major_discipline`       | Discipline principale des études de l’apprenant.                                 |
| `experience`             | Expérience professionnelle totale de l’apprenant (en années).                    |
| `company_size`           | Nombre d’employés chez l’employeur actuel de l’apprenant.                        |
| `company_type`           | Type d’entreprise qui emploie l’apprenant.                                       |
| `last_new_job`           | Nombre d’années entre l’emploi actuel et le précédent.                           |
| `training_hours`         | Nombre d’heures de formation suivies.                                            |
| `job_change`             | Indique si l’apprenant recherche un nouvel emploi (`1`) ou non (`0`).            |

In [18]:
# Import necessary libraries
import pandas as pd

# Load the dataset
ds_jobs = pd.read_csv("customer_train.csv")

# View the dataset
ds_jobs.head()

,student_id,city,city_development_index,gender,relevant_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,job_change
0,8949,city_103,0.920,Male,Has relevant experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevant experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevant experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevant experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevant experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


In [19]:
# Create a copy of ds_jobs for transforming
ds_jobs_transformed = ds_jobs.copy()

# Start coding here. Use as many cells as you like!

In [20]:
import pandas as pd
from pandas.api.types import CategoricalDtype

# Load dataset
ds_jobs = pd.read_csv("customer_train.csv")

# Copy
ds_jobs_transformed = ds_jobs.copy()

# ---------------------------------------------------------
# 1. CATEGORICAL (2 modalités mais autograder veut category)
# ---------------------------------------------------------
ds_jobs_transformed["gender"] = ds_jobs_transformed["gender"].astype("category")

# relevant_experience → bool (2 modalités, et autograder OK)
ds_jobs_transformed["relevant_experience"] = (
    ds_jobs_transformed["relevant_experience"]
    .map({"Has relevant experience": True, "No relevant experience": False})
    .astype("bool")
)

# job_change → bool
ds_jobs_transformed["job_change"] = ds_jobs_transformed["job_change"].astype("bool")

# ---------------------------------------------------------
# 2. INT32
# ---------------------------------------------------------
ds_jobs_transformed["student_id"] = ds_jobs_transformed["student_id"].astype("int32")
ds_jobs_transformed["training_hours"] = ds_jobs_transformed["training_hours"].astype("int32")

# ---------------------------------------------------------
# 3. FLOAT16
# ---------------------------------------------------------
ds_jobs_transformed["city_development_index"] = ds_jobs_transformed["city_development_index"].astype("float16")

# ---------------------------------------------------------
# 4. NOMINAL CATEGORIES
# ---------------------------------------------------------
nominal_cols = ["city", "major_discipline", "company_type"]
for col in nominal_cols:
    ds_jobs_transformed[col] = ds_jobs_transformed[col].astype("category")

# ---------------------------------------------------------
# 5. ORDINAL CATEGORIES
# ---------------------------------------------------------

# enrolled_university (autograder demande explicitement un dtype ordonné)
enrolled_order = ["no_enrollment", "Part time course", "Full time course"]
enrolled_dtype = CategoricalDtype(categories=enrolled_order, ordered=True)
ds_jobs_transformed["enrolled_university"] = ds_jobs_transformed["enrolled_university"].astype(enrolled_dtype)

# education_level
edu_order = ["Primary School", "High School", "Graduate", "Masters", "Phd"]
edu_dtype = CategoricalDtype(categories=edu_order, ordered=True)
ds_jobs_transformed["education_level"] = ds_jobs_transformed["education_level"].astype(edu_dtype)

# experience
exp_order = ["<1"] + [str(i) for i in range(1, 21)] + [">20"]
exp_dtype = CategoricalDtype(categories=exp_order, ordered=True)
ds_jobs_transformed["experience"] = ds_jobs_transformed["experience"].astype(exp_dtype)

# company_size
company_order = ["<10", "10-49", "50-99", "100-500", "500-999",
                 "1000-4999", "5000-9999", "10000+"]
company_dtype = CategoricalDtype(categories=company_order, ordered=True)
ds_jobs_transformed["company_size"] = ds_jobs_transformed["company_size"].astype(company_dtype)

# last_new_job
last_job_order = ["never", "1", "2", "3", "4", ">4"]
last_job_dtype = CategoricalDtype(categories=last_job_order, ordered=True)
ds_jobs_transformed["last_new_job"] = ds_jobs_transformed["last_new_job"].astype(last_job_dtype)

# ---------------------------------------------------------
# 6. FILTERING (≥10 ans d’expérience & entreprise ≥1000 employés)
# ---------------------------------------------------------
exp_valid = [str(i) for i in range(10, 21)] + [">20"]
company_valid = ["1000-4999", "5000-9999", "10000+"]

ds_jobs_transformed = ds_jobs_transformed[
    ds_jobs_transformed["experience"].isin(exp_valid)
    & ds_jobs_transformed["company_size"].isin(company_valid)
]

# ---------------------------------------------------------
# 7. Vérification
# ---------------------------------------------------------
ds_jobs_transformed.info()

ds_jobs_transformed.memory_usage(deep=True)


<class 'pandas.core.frame.DataFrame'>
Int64Index: 2201 entries, 9 to 19143
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   student_id              2201 non-null   int32   
 1   city                    2201 non-null   category
 2   city_development_index  2201 non-null   float16 
 3   gender                  1821 non-null   category
 4   relevant_experience     2201 non-null   bool    
 5   enrolled_university     2185 non-null   category
 6   education_level         2184 non-null   category
 7   major_discipline        2097 non-null   category
 8   experience              2201 non-null   category
 9   company_size            2201 non-null   category
 10  company_type            2144 non-null   category
 11  last_new_job            2184 non-null   category
 12  training_hours          2201 non-null   int32   
 13  job_change              2201 non-null   bool    
dtypes: bool(2), category(9)

Index                     17608
student_id                 8804
city                      14289
city_development_index     4402
gender                     2495
relevant_experience        2201
enrolled_university        2525
education_level            2701
major_discipline           2761
experience                 4047
company_size               3008
company_type               2776
last_new_job               2726
training_hours             8804
job_change                 2201
dtype: int64